In [0]:
%pip install databricks-feature-engineering

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

dates = ['2018-08', '2018-09', '2018-10', '2018-11', '2018-12', '2019-01', '2019-02', '2019-03', '2019-04', '2019-05', '2019-06', '2019-07', '2019-08', '2019-09', '2019-10', '2019-11', '2019-12', '2020-01', '2020-02', '2020-03', '2020-04', '2020-05', '2020-06', '2020-07', '2020-08', '2020-09', '2020-10', '2020-11', '2020-12', '2021-01', '2021-02', '2021-03', '2021-04', '2021-05', '2021-06']

In [0]:
def update_feature_store(catalog, database, table, query_file, dates):
    tablename = f"{catalog}.{database}.{table}"

    with open(query_file, "r") as f:
        query = f.read()

    fe = FeatureEngineeringClient()

    def table_exist(catalog, database, table):
        return spark.sql(
            f"SHOW TABLES FROM {catalog}.{database} LIKE '{table}'"
        ).count() > 0

    if table_exist(catalog, database, table):
        for dt_ref in dates:
            print(f"Escrevendo {dt_ref}")
            df = spark.sql(
                query.format(dt_ref=f"{dt_ref}-01")
            )
            fe.write_table(
                name=tablename,
                df=df,
                mode="merge"
            )
        print("Table updated")
    else:
        first_date = dates[0]
        df = spark.sql(
            query.format(dt_ref=f"{first_date}-01")
        )
        fe.create_table(
            name=tablename,
            primary_keys=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"],
            partition_columns=["DATA_REF"],
            df=df
        )
        print("Table created"
        )
        for dt_ref in dates[1:]:
            print(f"Escrevendo {dt_ref}")
            df = spark.sql(
                query.format(dt_ref=f"{dt_ref}-01")
            )
            fe.write_table(
                name=tablename,
                df=df,
                mode="merge"
            )
        print("Table created")

In [0]:
update_feature_store(
    catalog="feature_store",
    database="credit_score",
    table="fs_cadastral",
    query_file="fs_cadastral.sql",
    dates=dates
)

In [0]:
update_feature_store(
    catalog="feature_store",
    database="credit_score",
    table="fs_temporal",
    query_file="fs_temporal.sql",
    dates=dates
)

In [0]:
update_feature_store(
    catalog="feature_store",
    database="credit_score",
    table="fs_historico_financeiro",
    query_file="fs_historico_financeiro.sql",
    dates=dates
)